# RAG with Vector Database and Reranking on the OrchestrAI Synthetic Enterprise Corpus

This notebook extends the vector-database RAG pipeline with a reranking stage while preserving the overall retrieval and generation logic.

The goal is to evaluate the effect of reranking in isolation. Compared with the original `rag_vectordb` notebook, the main additions are:
- richer document processing with **Docling**
- tokenizer-aware chunking with **HybridChunker**
- persistent storage in **Milvus**
- a **cross-encoder reranker** placed after retrieval
- explicit source metadata for cleaner citations and inspection

This setup is useful for fair comparison because indexing, prompting, and generation remain broadly the same, while the retrieved candidate set is refined before it is passed to the LLM.


## What changes compared with the other notebooks?

### `rag_basic`
- direct DOCX conversion
- simpler splitting and indexing
- in-memory retrieval
- no persistent vector store

### `rag_vectordb`
- richer conversion and chunking
- persistent retrieval with Milvus
- improved source metadata handling

### `rag_reranker`
- keeps the same vector-database backbone as `rag_vectordb`
- retrieves a larger candidate set first
- reranks those candidates with a cross-encoder
- passes only the reranked top documents to the prompt builder

In other words, this notebook is designed to test whether reranking improves answer quality beyond dense retrieval alone.


## Environment requirements

This notebook assumes:
- Python **3.12** in the project virtual environment
- Ollama is running locally at `http://localhost:11434`
- the Ollama embedding model `mxbai-embed-large` is available
- the Ollama chat model `llama3.1` is available
- a **Milvus server** is running at `http://localhost:19530`
- the OrchestrAI corpus is stored under `./dummy_data`
- the package `sentence-transformers` is installed for the reranker

Run the optional install cell once if packages are missing.


## Imports and runtime checks

In [1]:
import sys
import warnings
import time
from pathlib import Path
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

warnings.filterwarnings("ignore")

print("Python:", sys.executable)
print("Ollama tags status:", requests.get("http://localhost:11434/api/tags").status_code)

import importlib
import results_logger
importlib.reload(results_logger)
from results_logger import save_result_row, clear_results_for_implementation
clear_results_for_implementation("reranker")

from haystack import Pipeline
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.rankers import SentenceTransformersSimilarityRanker

from milvus_haystack import MilvusDocumentStore
from milvus_haystack.milvus_embedding_retriever import MilvusEmbeddingRetriever

from haystack_integrations.components.embedders.ollama import OllamaDocumentEmbedder, OllamaTextEmbedder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

from haystack_integrations.components.converters.docling import DoclingConverter
from docling.chunking import HybridChunker

print("Imports work")

Python: /Users/nikos/Thesis/.venv/bin/python
Ollama tags status: 200
Cleared existing rows for implementation='reranker'
Imports work


## Connect to Milvus

This notebook uses a Milvus server connection rather than Milvus Lite.

Using Milvus here keeps the notebook aligned with a realistic vector-database deployment and makes the comparison with the original `rag_vectordb` notebook more meaningful.


In [2]:
document_store = MilvusDocumentStore(
    connection_args={"uri": "http://localhost:19530"},
    drop_old=True,
)

print("Milvus ready")

Milvus ready


## Load the OrchestrAI dataset

The corpus is organized by department under `./dummy_data`, and all `.docx` files are loaded recursively.

Each file is later converted, chunked, enriched with metadata, embedded, and written to Milvus.


In [ ]:
DOCUMENTS_DIR = Path("./dummy_data")
FILES = [str(f.resolve()) for f in DOCUMENTS_DIR.rglob("*.docx")]

print("Files found:", len(FILES))
for f in FILES[:10]:
    print("-", Path(f).name)

Files found: 54
- VAL-SALES-003_Campaign_Approval_Workflow_refreshed.docx
- VAL-SALES-004_CRM_Field_Governance_Note_refreshed.docx
- VAL-SALES-005_Discount_Exception_Policy_refreshed.docx
- VAL-SALES-001_Lead_Routing_Rules_refreshed.docx
- VAL-SALES-002_Webinar_Launch_Brief_refreshed.docx
- VAL-SEC-003_Patch_Compliance_Reminder.docx
- VAL-SEC-004_Data_Retention_Notice.docx
- VAL-SEC-005_Third_Party_Access_Approval_Standard.docx
- VAL-SEC-002_Privileged_Access_Review_Memo.docx
- VAL-SEC-001_Phishing_Alert_Bulletin.docx


## Indexing strategy

To keep retrieval reliable and source-aware, the indexing flow is:

1. Convert each file with `DoclingConverter`
2. Attach explicit source metadata such as `file_name`, `file_path`, and `department`
3. Apply an additional `DocumentSplitter` before embedding
4. Remove metadata fields that Milvus cannot store cleanly
5. Generate embeddings with Ollama
6. Write the embedded chunks to Milvus

This file-by-file indexing approach also makes it easier to inspect retrieved chunks and avoid vague `unknown source` outputs during answer generation.


In [4]:
OLLAMA_EMBED_MODEL = "mxbai-embed-large"
HF_TOKENIZER_ID = "mixedbread-ai/mxbai-embed-large-v1"

converter = DoclingConverter(
    chunker=HybridChunker(
        tokenizer=HF_TOKENIZER_ID,
        max_tokens=300,
    )
)

splitter = DocumentSplitter(
    split_by="word",
    split_length=80,
    split_overlap=20
)

doc_embedder = OllamaDocumentEmbedder(
    model=OLLAMA_EMBED_MODEL,
    url="http://localhost:11434"
)

writer = DocumentWriter(document_store=document_store)

def normalize_meta(meta):
    cleaned = {}
    for key, value in (meta or {}).items():
        if key == "_split_overlap":
            continue
        if isinstance(value, (str, int, float, bool)) or value is None:
            cleaned[key] = value
        else:
            cleaned[key] = str(value)
    return cleaned

stored_chunks = 0
indexing_start = time.perf_counter()

for i, path in enumerate(FILES, 1):
    print(f"[{i}/{len(FILES)}] Indexing {Path(path).name}")

    converted = converter.run(paths=[path])["documents"]

    for doc in converted:
        meta = normalize_meta(doc.meta)
        meta["file_path"] = path
        meta["file_name"] = Path(path).name
        meta["department"] = Path(path).parent.name
        doc.meta = meta

    split_docs = splitter.run(documents=converted)["documents"]

    for doc in split_docs:
        meta = normalize_meta(doc.meta)
        meta["file_path"] = path
        meta["file_name"] = Path(path).name
        meta["department"] = Path(path).parent.name
        doc.meta = meta

    embedded_docs = doc_embedder.run(documents=split_docs)["documents"]
    writer.run(documents=embedded_docs)

    stored_chunks += len(embedded_docs)

indexing_time_s = round(time.perf_counter() - indexing_start, 4)

print("Indexing completed.")
print("Stored chunks in Milvus:", stored_chunks)
print("Document store count:", document_store.count_documents())
print("Indexing time (s):", indexing_time_s)

[1/54] Indexing VAL-SALES-003_Campaign_Approval_Workflow_refreshed.docx


Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.77it/s]


[2/54] Indexing VAL-SALES-004_CRM_Field_Governance_Note_refreshed.docx


Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.99it/s]


[3/54] Indexing VAL-SALES-005_Discount_Exception_Policy_refreshed.docx


...

[53/54] Indexing TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx


Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.46it/s]


[54/54] Indexing TEAMQA-PROD-001_OrchestrAI_Product_and_Program_Management_Handbook.docx


Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.44s/it]

Indexing completed.
Stored chunks in Milvus: 654
Document store count: 654
Indexing time (s): 34.6704


## Build the reranking RAG pipeline

The answer-generation logic stays aligned with the earlier notebooks:
- use only the retrieved context
- avoid inventing missing information
- cite source filenames when possible

The main architectural difference is the added reranking step:

`text_embedder -> retriever -> reranker -> prompt_builder -> llm`

This keeps the experiment fair: the pipeline still uses the same corpus, vector store, and generator, but now refines retrieved candidates before prompting the LLM.


In [5]:
template = [
    ChatMessage.from_user(
        """
Respond to the User Query using only the provided Context.

General Guidelines:
- Use only the provided context.
- If the answer is not found in the context, say so clearly.
- Do not make up facts, document sections, pages, or citations.
- If the answer comes from multiple documents, cite all relevant documents.
- Respond in the same language as the user’s query.
- Do not use emojis.
- Be professional and concise.
- Do not write a conclusion or follow-up unless the user asks for it.
- If the context contains a direct policy rule, exception, SLA, or escalation instruction, state it directly and do not hedge.
- Prefer the most specific procedural source over broader handbook summaries when both are present.
- If the retrieved context does not explicitly contain the rule, exception, owner, or next step needed to answer the question, say that the retrieved context is insufficient instead of inferring or guessing.
- Do not mention system names, tools, workflows, or teams unless they are explicitly present in the retrieved context.

Citation Rules:
- For every important claim, cite the source document filename.
- If available in the retrieved context, also mention the document ID or department.
- Do not mention source chapter, section, or page unless they are explicitly present in the retrieved context.

Context:
{% for document in documents %}
[Source: {{ document.meta.get("file_name", document.meta.get("file_path", "unknown source")) }}]
{{ document.content }}

{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

text_embedder = OllamaTextEmbedder(
    model=OLLAMA_EMBED_MODEL,
    url="http://localhost:11434"
)

retriever = MilvusEmbeddingRetriever(
    document_store=document_store,
    top_k=12
)

reranker = SentenceTransformersSimilarityRanker(
    model=RERANK_MODEL,
    top_k=3
)

chat_generator = OllamaChatGenerator(
    model="llama3.1",
    url="http://localhost:11434",
    generation_kwargs={
        "temperature": 0.0,
        "top_p": 1.0,
        "seed": 42,
    }
)

prompt_builder = ChatPromptBuilder(
    template=template,
    required_variables=["question", "documents"]
)

reranker.warm_up()

advanced_rag_pipeline = Pipeline()
advanced_rag_pipeline.add_component("text_embedder", text_embedder)
advanced_rag_pipeline.add_component("retriever", retriever)
advanced_rag_pipeline.add_component("reranker", reranker)
advanced_rag_pipeline.add_component("prompt_builder", prompt_builder)
advanced_rag_pipeline.add_component("llm", chat_generator)

advanced_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
advanced_rag_pipeline.connect("retriever.documents", "reranker.documents")
advanced_rag_pipeline.connect("reranker.documents", "prompt_builder.documents")
advanced_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")


The CrossEncoder `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.
Loading weights: 100%|██████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 9202.26it/s]


🚅 Components
  - text_embedder: OllamaTextEmbedder
  - retriever: MilvusEmbeddingRetriever
  - reranker: SentenceTransformersSimilarityRanker
  - prompt_builder: ChatPromptBuilder
  - llm: OllamaChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> reranker.documents (List[Document])
  - reranker.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

## Helper function for evaluation queries

The function below:
- embeds the user question
- retrieves an initial candidate set from Milvus
- reranks those candidates with the cross-encoder
- prints the reranked sources and short snippets
- runs the full pipeline
- prints the final answer
- returns both the reranked documents and the final answer

This makes it easier to compare retrieval behavior and final responses across `rag_basic`, `rag_vectordb`, and `rag_reranker`.


In [6]:
def run_reranker_query(question, retriever_top_k=12, reranker_top_k=3):
    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("=" * 80)

    retrieval_start = time.perf_counter()

    q_emb = text_embedder.run(text=question)["embedding"]

    retr_out = retriever.run(query_embedding=q_emb, top_k=retriever_top_k)
    candidate_docs = retr_out["documents"]

    rerank_out = reranker.run(
        query=question,
        documents=candidate_docs,
        top_k=reranker_top_k,
    )
    top_docs = rerank_out["documents"]

    retrieval_time_s = round(time.perf_counter() - retrieval_start, 4)

    print(
        f"\nRERANKED DOCUMENTS: {len(top_docs)} "
        f"(from {len(candidate_docs)} retrieved candidates)"
    )
    print(f"RETRIEVAL + RERANK TIME (s): {retrieval_time_s}\n")

    for i, d in enumerate(top_docs, 1):
        source = d.meta.get("file_name", d.meta.get("file_path", "unknown source"))
        department = d.meta.get("department", "unknown department")
        snippet = (d.content or "")[:400].replace("\n", " ")
        print(f"[{i}] Source: {source} | Department: {department}")
        print(f"    {snippet}...")
        print()

    generation_start = time.perf_counter()

    result = advanced_rag_pipeline.run({
        "text_embedder": {"text": question},
        "retriever": {"top_k": retriever_top_k},
        "reranker": {"query": question, "top_k": reranker_top_k},
        "prompt_builder": {"question": question},
    })

    answer = result["llm"]["replies"][0].text
    generation_time_s = round(time.perf_counter() - generation_start, 4)

    print("=" * 80)
    print("FINAL ANSWER:")
    print("=" * 80)
    print(answer)
    print(f"\nGENERATION TIME (s): {generation_time_s}")

    return top_docs, answer, retrieval_time_s, generation_time_s

In [7]:
def run_and_save_reranker(question, scenario_id, retriever_top_k=12, reranker_top_k=3):
    top_docs, answer, retrieval_time_s, generation_time_s = run_reranker_query(
        question,
        retriever_top_k=retriever_top_k,
        reranker_top_k=reranker_top_k,
    )

    save_result_row(
        implementation="reranker",
        scenario_id=scenario_id,
        question=question,
        generated_answer=answer,
        top_docs=top_docs,
        retriever_top_k=retriever_top_k,
        reranker_top_k=reranker_top_k,
        indexing_time_s=indexing_time_s,
        retrieval_time_s=retrieval_time_s,
        generation_time_s=generation_time_s,
    )

    return top_docs, answer

## Evaluation scenarios

The following five questions are used as progressively more demanding evaluation scenarios:

1. Basic fact retrieval
2. Procedural retrieval
3. Multi-team operational reasoning
4. Policy and exception handling
5. Cross-functional enterprise reasoning

Together, these scenarios help test not only retrieval accuracy, but also whether the model can use the retrieved context to produce grounded, source-based answers of increasing complexity.


### Scenario 1 — Basic fact retrieval

This is the clean baseline scenario. It checks whether the system can retrieve and answer a single operational fact accurately.

- Retrieval type: single-document, single-fact lookup
- Main test: exact fact retrieval
- Likely supporting source: Customer Success / Support SLA material


In [8]:
top_docs_s1, answer_s1 = run_and_save_reranker(
    'What is the first-response SLA for a Sev-1 support incident?',
    scenario_id="r1",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
What is the first-response SLA for a Sev-1 support incident?

RERANKED DOCUMENTS: 3 (from 12 retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.5135

[1] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Specialised Q&A **Q: Who owns customer communication during a Sev-1 incident?** **A:** The assigned support lead owns the operational updates, but the Customer Success Manager owns stakeholder alignment and executive communication. If the issue touches release risk or hardware failure, Engineering or Logistics may join the bridge. **Q: What is the target first...

[2] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Service targets and escalation guide Sev-1 first response, **Target** = 15 min. Sev-1 first response, **Reviewed by** = Support manager. Sev-1 first response, **Notes** = Cl

### Scenario 2 — Procedural retrieval

This scenario is slightly more complex because the answer is not a single fact but a short operational process.

- Retrieval type: checklist or procedure retrieval
- Main test: whether the system can recover and summarize a structured process
- Likely supporting source: onboarding or enablement documentation


In [9]:
top_docs_s2, answer_s2 = run_and_save_reranker(
    'What steps must be completed before a new employee is considered ready for day one at OrchestrAI?',
    scenario_id="r2",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
What steps must be completed before a new employee is considered ready for day one at OrchestrAI?

RERANKED DOCUMENTS: 3 (from 12 retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.3328

[1] Source: VAL-HR-001_Day_One_Onboarding_Checklist.docx | Department: HR
    Day-One Onboarding Checklist **Document ID:** VAL-HR-001 **Owner:** HR & People Operations **Confidentiality:** Internal Use Only Purpose: This checklist standardizes the minimum tasks that must be complete before an OrchestrAI employee can be considered ready for day-one onboarding. **Department**, **Value** = HR & People Operations. **Effective date**, **Value** = 2026-02-24. **Applies to**, **V...

[2] Source: VAL-HR-001_Day_One_Onboarding_Checklist.docx | Department: HR
    Day-One Onboarding Checklist Day-one completion rules **•** The new hire must be able to sign in with baseline credentials and MFA. **•** Payroll and tax forms must be complete or formally escalated. **•** The manager must confirm first-wee

### Scenario 3 — Multi-team operational reasoning

This scenario requires combining information from more than one team to answer a realistic workplace problem.

- Retrieval type: multi-document, cross-team retrieval
- Main test: whether the system can synthesize HR, IT, and access-related responsibilities
- Likely supporting sources: onboarding, IT, and security guidance


In [10]:
top_docs_s3, answer_s3 = run_and_save_reranker(
    'A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?',
    scenario_id="r3",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?

RERANKED DOCUMENTS: 3 (from 12 retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.3537

[1] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    Common Questions **Q: How far in advance should a laptop be requested?** A: Standard new-hire laptop requests should be present in the hiring workflow at least seven business days before the employee start date. International shipments, executive hires, and custom software bundles may require ten to fifteen business days. **Q: Which team starts the request?** A: The request begins with the hiring ...

[2] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    1 business day. Non-standard software or admin rights request, **Primary Owner** = Endpoint Engineering. Non-standard software or admin rights request, **Escalation Trigger** = Securit

### Scenario 4 — Policy and exception handling

This scenario checks whether the system can distinguish the normal rule from the emergency exception path.

- Retrieval type: policy retrieval with exception logic
- Main test: whether the system can identify when a standard workflow may be bypassed
- Likely supporting source: procurement, operations, or incident-response policy


In [11]:
top_docs_s4, answer_s4 = run_and_save_reranker(
    'Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?',
    scenario_id="r4",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?

RERANKED DOCUMENTS: 3 (from 12 retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.1834

[1] Source: TEAMQA-SALES-001_OrchestrAI_Sales_and_Solutions_Handbook.docx | Department: Q&A Hanbooks
    opportunity has a documented warehouse problem, a credible business sponsor, a timeline that matches operational constraints, a measurable business case, and enough process detail to determine whether OrchestrAI can support the workflow without hidden delivery risk. **Q: When must a Solutions Consultant join a deal?** **A:** A Solutions Consultant must join when the opportunity involves integratio...

[2] Source: TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx | Department: Q&A Hanbooks
    OrchestrAI Logistics and Field Fulfillment Handbook 2. Team Mission and Operating Context The Logistics and Field Fulfillment team ensures that Orchest

### Scenario 5 — Cross-functional enterprise reasoning

This is the most demanding scenario in the notebook because it requires cross-functional reasoning across several business areas.

- Retrieval type: multi-hop, multi-document retrieval
- Main test: whether the system can combine support, product, logistics, and commercial risk signals into one grounded answer
- Likely supporting sources: customer success, product, support, and logistics material


In [12]:
top_docs_s5, answer_s5 = run_and_save_reranker(
    'A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?',
    scenario_id="r5",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?

RERANKED DOCUMENTS: 3 (from 12 retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.1953

[1] Source: VAL-CS-004_Renewal_Risk_Signals_Brief.docx | Department: Customer Support
    Renewal Risk Signals Brief **Document ID:** VAL-CS-004 **Department:** Customer Success & Support **Confidentiality:** Internal Use Only **Effective date**, Director, Customer Success Operations = 2026-02-10. **Effective date**, **Document type** = **Review date**. **Effective date**, Risk brief = 2026-08-10 Purpose: Defines the most important signals used by OrchestrAI Customer Success teams to i...

[2] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    every escalation to Engineering. **Q: What creates renewal risk?** **A:** Common 